# Лабораторная работа № 5. Деревья решений и ансамбли

**Курс:** Классическое машинное обучение, 4 курс прикладной математики

## Цель работы

Реализовать дерево решений, изучить ансамблевые методы: случайный лес и градиентный бустинг.

**Используемые инструменты:** `numpy`, `sklearn.tree`, `sklearn.ensemble`, `xgboost` (опционально), `matplotlib`.

### Регламент сдачи

Работа сдаётся в виде этого же ноутбука, дополненного вашим кодом. Обязательно:

1. Читаемый код с комментариями.
2. Визуализации (графики, таблицы).
3. **Текстовый вывод после каждого задания** — не только код, но и объяснение результата.
4. Финальный вывод по работе.

**Критерии оценки:** корректность реализации — 30 %, качество визуализаций и анализа — 20 %,
обоснованность выводов — 20 %, сравнение с эталонными реализациями — 15 %,
оригинальность и дополнительная работа — 15 %.

> Ячейки, помеченные `# TODO`, нужно заполнить самостоятельно.
> Ячейки с готовым кодом можно просто выполнить — они подготавливают данные и графики.

## Подготовка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
sns.set_palette("viridis")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, roc_auc_score

diabetes = load_diabetes()
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=RANDOM_STATE)

print("Регрессия:", Xr_train.shape)

## Задание 1. Класс `DecisionTreeRegressor`

Дерево строится рекурсивно. В каждом узле перебираем все признаки и все пороги,
выбирая расщепление с максимальным уменьшением ошибки:

$$Q(R, j, t) = \frac{|R_\ell|}{|R|}\,\mathrm{MSE}(R_\ell) + \frac{|R_r|}{|R|}\,\mathrm{MSE}(R_r) \to \min_{j, t},$$

где $\mathrm{MSE}(R) = \frac{1}{|R|}\sum_{i \in R}(y_i - \bar{y}_R)^2$. В листе предсказание — среднее $\bar{y}$.

Критерии остановки: достигнута `max_depth`, в узле меньше `min_samples_split` объектов,
или расщепление не уменьшает ошибку.

In [ ]:
class Node:
    """Узел дерева: либо лист со значением, либо предикат [feature] <= threshold."""

    def __init__(self, value=None, feature=None, threshold=None, left=None, right=None):
        self.value = value
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right

    @property
    def is_leaf(self):
        return self.value is not None


class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    @staticmethod
    def _mse(y):
        # TODO: np.mean((y - y.mean()) ** 2), но аккуратно с пустым массивом
        raise NotImplementedError

    def _best_split(self, X, y):
        """Возвращает (feature, threshold, выигрыш) или (None, None, 0)."""
        # TODO: переберите признаки и пороги (порогами удобно брать середины
        #       между соседними уникальными значениями), верните лучшее расщепление
        raise NotImplementedError

    def _build(self, X, y, depth=0):
        # TODO: рекурсия с проверкой критериев остановки
        raise NotImplementedError

    def fit(self, X, y):
        self.root = self._build(np.asarray(X), np.asarray(y))
        return self

    def _predict_one(self, x, node):
        # TODO: спуск по дереву до листа
        raise NotImplementedError

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in np.asarray(X)])

## Задание 2. Сравнение с `sklearn`

Обучите своё дерево и `sklearn.tree.DecisionTreeRegressor` с теми же гиперпараметрами
на `diabetes`, сравните MSE. Расхождение в пределах пары процентов — норма
(sklearn использует другие правила выбора порогов).

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# TODO: обучите обе модели для max_depth = 1..10,
#       постройте график MSE(глубина) на train и test для обеих реализаций

**Вывод:** *при какой глубине начинается переобучение? Как это видно на графике?*

## Задание 3. Случайный лес

Бэггинг: обучаем $B$ деревьев на бутстрап-выборках и усредняем ответы. Дисперсия ансамбля
падает примерно в $B$ раз, если деревья некоррелированы — поэтому случайный лес дополнительно
случайно ограничивает набор признаков в каждом узле (`max_features`).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: постройте график MSE на тесте в зависимости от n_estimators (1..300).
#       На том же графике горизонтальной линией отметьте MSE одиночного дерева.
#       Где кривая выходит на плато?

In [ ]:
# TODO: исследуйте max_features ("sqrt", "log2", 0.3, 1.0) — как меняется качество?
#       Объясните результат через компромисс «смещение — разброс».

## Задание 4. Лес против бустинга на классификации

Сравните на `breast_cancer` по accuracy и AUC:

* `RandomForestClassifier`,
* `GradientBoostingClassifier` (или `XGBClassifier`, если установлен `xgboost`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

cancer = load_breast_cancer()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, stratify=cancer.target, random_state=RANDOM_STATE)

# Если установлен xgboost — используйте его, иначе GradientBoostingClassifier
try:
    from xgboost import XGBClassifier
    booster = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)
    print("используем XGBoost")
except ImportError:
    booster = GradientBoostingClassifier(random_state=RANDOM_STATE)
    print("xgboost не установлен, используем GradientBoostingClassifier")

# TODO: обучите обе модели, сравните accuracy и AUC, замерьте время обучения (time.perf_counter)

## Задание 5. Важность признаков

Постройте столбчатые диаграммы важностей для леса и бустинга.
Дополнительно посчитайте `permutation_importance` — она честнее для скоррелированных признаков.

In [ ]:
from sklearn.inspection import permutation_importance

# TODO: сравните feature_importances_ и permutation_importance.
#       Совпадают ли топ-5 признаков? Если нет — объясните расхождение.

## Дополнительное задание. Cost-complexity pruning

Реализуйте обрезку своего дерева по критерию $R_\alpha(T) = R(T) + \alpha|T|$,
где $|T|$ — число листьев. Сравните с `DecisionTreeRegressor(ccp_alpha=...)`
и постройте график «качество — число листьев».

In [ ]:
# TODO: постройте путь обрезки (cost_complexity_pruning_path) для sklearn-дерева
#       и сравните с вашей реализацией

## Финальный вывод

*Напишите здесь связный вывод по работе (5–10 предложений):*

- какие методы вы применили и почему;
- какие результаты получили в числах;
- где реализация «с нуля» разошлась с эталоном из `sklearn` и в чём причина;
- что бы вы улучшили, будь у вас больше времени.